# 🔧 Fix: Injeção de Contexto no Dino SDK

## 🎯 Problema Identificado:
- ✅ **Notebook de teste direto FUNCIONOU** - dbutils está OK
- ❌ **Dino SDK FALHOU** - `globals()` não vê dbutils quando SDK é importado

## 💡 Solução:
**Injetar as credenciais extraídas diretamente no SDK** ao invés de depender do globals().

### Estratégia:
1. Extrair credenciais diretamente (sabemos que funciona)
2. Injetar no SDK antes de usar
3. Validar funcionamento completo

In [ ]:
# Passo 1: Extrair credenciais (sabemos que funciona)
print("🔍 Passo 1: Extrair Credenciais Diretamente")
print("=" * 45)

# Método que sabemos que funciona
try:
    # Verificar dbutils
    print(f"dbutils tipo: {type(dbutils)}")
    print("✅ dbutils está disponível")
    
    # Extrair workspace URL
    databricks_instance = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
    workspace_url = f"https://{databricks_instance}" if not databricks_instance.startswith('https://') else databricks_instance
    print(f"✅ Workspace URL: {workspace_url}")
    
    # Extrair token
    admin_token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
    masked_token = f"{admin_token[:15]}...{admin_token[-8:]}"
    print(f"✅ Admin Token: {masked_token}")
    
    # Salvar credenciais
    extracted_credentials = {
        'workspace_url': workspace_url,
        'admin_token': admin_token,
        'dbutils': dbutils
    }
    
    print("\n🎯 Credenciais extraídas com sucesso!")
    
except Exception as e:
    print(f"❌ Erro na extração: {e}")
    extracted_credentials = None

In [ ]:
# Passo 2: Configurar variáveis de ambiente para forçar o SDK a usar
print("🔧 Passo 2: Configurar Variáveis de Ambiente")
print("=" * 45)

if extracted_credentials:
    import os
    
    # Configurar variáveis de ambiente
    os.environ['DATABRICKS_HOST'] = extracted_credentials['workspace_url']
    os.environ['DATABRICKS_TOKEN'] = extracted_credentials['admin_token']
    
    print(f"✅ DATABRICKS_HOST configurado: {extracted_credentials['workspace_url']}")
    print(f"✅ DATABRICKS_TOKEN configurado: {extracted_credentials['admin_token'][:15]}...")
    
    # Verificar se foram definidas
    print(f"\n🔍 Verificação:")
    print(f"   DATABRICKS_HOST: {os.getenv('DATABRICKS_HOST')}")
    print(f"   DATABRICKS_TOKEN: {os.getenv('DATABRICKS_TOKEN', 'Not Set')[:15]}...")
    
else:
    print("❌ Credenciais não disponíveis para configurar")

In [ ]:
# Passo 3: Testar Dino SDK com env vars configuradas
print("🦕 Passo 3: Testar Dino SDK com ENV VARS")
print("=" * 42)

try:
    # Importar após configurar env vars
    from src.keyvault_config import KeyVaultConfigManager
    
    print("✅ Dino SDK importado")
    
    # Criar manager
    kv_manager = KeyVaultConfigManager(
        keyvault_name="dino-shared-keyvault",
        catalog_name="dino_catalog",
        schema_name="test_env_vars_fix"
    )
    
    print("✅ KeyVaultConfigManager criado")
    
    # Forçar inicialização com env vars (pular detecção de contexto)
    print("\n🔧 Inicializando cliente...")
    kv_manager._initialize_databricks_client()
    
    # Verificar resultado
    if hasattr(kv_manager, 'databricks_client') and kv_manager.databricks_client:
        print("✅ Cliente Databricks inicializado!")
        
        # Testar operação
        current_user = kv_manager.databricks_client.current_user.me()
        print(f"👤 Usuário: {current_user.user_name}")
        
        print("\n🎉 SUCESSO! SDK funcionou com env vars")
        
    else:
        print("❌ Cliente não foi inicializado")
        
except Exception as e:
    print(f"❌ Erro no SDK: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# Passo 4: Alternativa - Injeção direta no SDK
print("💉 Passo 4: Injeção Direta no SDK")
print("=" * 35)

if extracted_credentials:
    try:
        # Método alternativo: injetar dbutils no globals do módulo SDK
        import src.keyvault_config as kv_module
        
        # Injetar dbutils no módulo
        kv_module.__dict__['dbutils'] = extracted_credentials['dbutils']
        print("✅ dbutils injetado no módulo SDK")
        
        # Testar função de contexto agora
        from src.keyvault_config import get_notebook_context
        
        context = get_notebook_context()
        
        if context:
            print("✅ get_notebook_context funcionou após injeção!")
            print(f"   workspace_url: {'✅' if context.get('workspace_url') else '❌'}")
            print(f"   token: {'✅' if context.get('token') else '❌'}")
            
            # Testar KeyVaultConfigManager novamente
            kv_manager2 = KeyVaultConfigManager(
                keyvault_name="dino-shared-keyvault",
                catalog_name="dino_catalog", 
                schema_name="test_injection_fix"
            )
            
            kv_manager2._initialize_databricks_client()
            
            if hasattr(kv_manager2, 'databricks_client') and kv_manager2.databricks_client:
                print("✅ SDK funcionou com injeção direta!")
                current_user = kv_manager2.databricks_client.current_user.me()
                print(f"👤 Usuário: {current_user.user_name}")
            else:
                print("❌ SDK ainda falhou com injeção")
                
        else:
            print("❌ get_notebook_context ainda falhou")
            
    except Exception as e:
        print(f"❌ Erro na injeção: {e}")
        import traceback
        traceback.print_exc()
else:
    print("❌ Credenciais não disponíveis para injeção")

In [ ]:
# Passo 5: Solução definitiva - Monkey patch
print("🐒 Passo 5: Monkey Patch - Solução Definitiva")
print("=" * 48)

if extracted_credentials:
    try:
        # Criar função de contexto personalizada
        def custom_get_notebook_context():
            return {
                'dbutils': extracted_credentials['dbutils'],
                'workspace_url': extracted_credentials['workspace_url'], 
                'token': extracted_credentials['admin_token']
            }
        
        # Substituir função no módulo
        import src.keyvault_config as kv_module
        kv_module.get_notebook_context = custom_get_notebook_context
        
        print("✅ Função get_notebook_context substituída")
        
        # Testar agora
        kv_manager3 = KeyVaultConfigManager(
            keyvault_name="dino-shared-keyvault",
            catalog_name="dino_catalog",
            schema_name="test_monkey_patch"
        )
        
        print("✅ Manager criado")
        
        # Inicializar
        kv_manager3._initialize_databricks_client()
        
        if hasattr(kv_manager3, 'databricks_client') and kv_manager3.databricks_client:
            print("🎉 SUCESSO TOTAL! Monkey patch funcionou!")
            current_user = kv_manager3.databricks_client.current_user.me()
            print(f"👤 Usuário: {current_user.user_name}")
            print(f"📧 Email: {current_user.emails[0].value if current_user.emails else 'N/A'}")
            
            # Testar operação de Secret Scope
            try:
                scopes = kv_manager3.databricks_client.secrets.list_scopes()
                print(f"📋 Secret Scopes: {len(scopes)} encontrados")
                
                print(f"\n🚀 SDK TOTALMENTE FUNCIONAL!")
                print(f"   ✅ Contexto extraído")
                print(f"   ✅ Cliente inicializado") 
                print(f"   ✅ API funcionando")
                print(f"   ✅ Secret Scopes acessíveis")
                
            except Exception as e:
                print(f"⚠️ Erro ao acessar Secret Scopes: {e}")
                
        else:
            print("❌ Cliente ainda não foi inicializado")
            
    except Exception as e:
        print(f"❌ Erro no monkey patch: {e}")
        import traceback
        traceback.print_exc()
else:
    print("❌ Credenciais não disponíveis")

## ✅ Resumo das Soluções Testadas

### 🎯 **Métodos Implementados:**

1. **🔧 Variáveis de Ambiente** - Forçar SDK a usar DATABRICKS_HOST/TOKEN
2. **💉 Injeção no Módulo** - Injetar dbutils diretamente no módulo SDK  
3. **🐒 Monkey Patch** - Substituir função get_notebook_context() completamente

### 📊 **Resultados:**
- Se **Método 1** funcionou: Use env vars como solução
- Se **Método 3** funcionou: Solução monkey patch é mais robusta
- Se **nenhum** funcionou: Problema mais profundo no ambiente

### 🚀 **Próximos Passos:**
- Implementar solução que funcionou no código oficial do SDK
- Gerar versão 1.1.2 com correção definitiva